**Notebook zum erstellen der Modelle**

In diesem Notebook werden die Modelle erstellt und angepasst.
Dafür werden bilder gefiltert und vorverarbeitet. 
Anschließend wird auf einem verringerten Datensatz ein Gridsearch algorithmus durchgeführt.
Die Modelle werden final mit dem gesamten Datensatz gefitted.

Das Notebook wurde nicht vollständig mit dem gesamten Datensatz durchlaufen, da einige Funktionen schlecht skalieren. Wenn ein Wechsel stattfindet zwischen der verringerten Datensatzgröße wird dies angemerkt.

Es ist nicht empfohlen, das Notebook mit dem vollen Datensatz komplett durchlaufen zu lassen, da dies einige Stunden dauert. 

Im ersten Block werden alle Imports getätigt. Die letzten vier Funktionen sind eigens erstellt für die Verarbeitung von Bildern. Sie wurden in eine Pythondatei verschoben um eine Wiederverwendung im anderen Notebook zu ermöglichen.

In [1]:
#import aller genereller Funktionen und Bibliotheken
import numpy as np
from pathlib import Path
from itertools import islice
from natsort import natsorted
from concurrent.futures import ThreadPoolExecutor
import pickle

#import aller ml spezifischen Bibliotheken
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

#import aller eigener Bildverarbeitenden Funktionen 
# (erleichtert Nutzung in mehreren Notebooks)
from imageFunctions import readImage
from imageFunctions import preprocessImage
from imageFunctions import processImage
from imageFunctions import detect_background_presence

Der folgende Block lädt die Daten in das Notebook. Die Dateien werden dabei mit anderer Sortierung geladen, als sie in den Variablen gespeichert werden. Die natürliche Sortierung in den Variablen wurde eingefügt, um das Debuggen zu erleichtern. 

Ersichtlich ist auch, dass die nicht weiter genutzten Variablen gelöscht werden. Dies wird an mehreren Stellen im Code getan, um den Speicherverbrauch zu begrenzen. Bei Nutzung des gesamten Datensatzes kann der benötigte Arbeitsspeicher dennoch ungefähr 12 GB betragen.

Die zu ändernden Werte zur Nutzung des gesamten oder nur eines Teildatensatzes sind hervorgehoben. Die funktionalität wurde zur besseren Lesbarkeit in eine Funktion integriert.

In [ ]:
currentPath = Path().cwd()

# diese Werte anpassen wenn nötig!!!
maxIms = 2500
useAllIms=False
#--------------

def readFolderAddLabel(path, label, numImages=maxIms, useAll=useAllIms):
    if useAll: numImages = len([f for f in path.iterdir() if f.is_file()])
    imPaths = natsorted([str(file) for file in islice(path.iterdir(), numImages) if file.is_file()])
    return [[imPath, label] for imPath in imPaths]

targetPath = currentPath.parent / 'data' / 'KGT_noDefect'
imsNoF = readFolderAddLabel(targetPath, False)

targetPath = currentPath.parent / 'data' / 'KGT_pitting'
imsF = readFolderAddLabel(targetPath, True)

ims = imsNoF + imsF
paths = [x[0] for x in ims]

del(imsNoF, imsF, currentPath, targetPath)

Im folgenden Block werden die Bilder mit Hintergrund detektiert. Dieser Schritt ist recht aufwändig und wurde daher parallelisiert. 

Die genutzten Funktionen sind in imageFunctions.py zu finden. Wie im Bericht beschrieben werden die Bilder geladen, ein lokaler Kontrastausgleich (CLAHE) und ein Resize wird angewandt und es wird ein Hintergrund erkannt, wenn helle Zonen an den oberen oder unteren, nicht aber an beide angrenzen.
Genauer ist dies im Bericht nachzulesen.

Die Namen der Bilder mit so detektiertem Hintergrund werden gespeichert. Anschließend wird eine Liste mit den Pfaden der Bilder erstellt, die nicht mit den gefundenen Namen übereinstimmen.

In [ ]:
def process_image_safe(image_path):
    try:
        img = readImage(image_path)
        pre = preprocessImage(img)
        if detect_background_presence(pre, visualize=False):
            return Path(image_path).name
    except Exception as e:
        print(f"Error: {e}")
    return None

with ThreadPoolExecutor() as executor:
    results = list(executor.map(process_image_safe, paths))

namesFilteredV2 = [r for r in results if r]
pathsFiltered = [x for x in paths if Path(x).name not in namesFilteredV2]

Der anschließende Block erstellt die Features für die jeweiligen Bilder, die nicht ausgefiltert wurden. Die Struktur der ims Variable ist nicht generell effizient oder nötig. Die genutzten Ressourcen zur erstellung und zerlegung in Einzelteile sind jedoch relatv gering. Daher wurde dies in der Bearbeitung toleriert. Ebenso werden die Bilder im vorangegangenen Block bereits eingelesen und vorverarbeitet. Diese Schritte müssten nicht doppelt vorgenommen werden, würde die Struktur des Notebooks noch einmal überarbeitet werden.

In [15]:
ims = [[processImage(preprocessImage(readImage(imgPath))),boolVal] 
       for imgPath, boolVal in ims 
       if imgPath in pathsFiltered]

Im folgenden Block werden die Daten in Zielvariable und Eingangsdaten aufgeteilt.

Die Features werden normalisiert und anhand einer Principal-Component Analyse reduziert. Dieser Schritt hat im Test mit dem kompletten Datensatz drei Stunden gedauert.

Der reduzierte Featuresatz wird anschließend genutzt, um mit den Zielvariablen zu einem Test- und einem Trainingsdatensatz geteilt zu werden mit einem Train_Test Split. Der Test datensatz umfasst dabei 15% der Datenpunkte.

In [16]:
images, labels = zip(*ims)
X = np.array(images)
y = np.array(labels)

scaler= StandardScaler().fit(X)
X_scaled = scaler.transform(X)
pca = PCA(n_components=0.95).fit(X_scaled)
X_pca = pca.transform(X_scaled)

XTrain, XTest, yTrain, yTest = train_test_split(X_pca,y, stratify=y, test_size=0.15, random_state=42)

Da das Modell auf einen reduzierten Featuresatz trainiert wird, braucht es auch weiterhin einen Featuresatz dieser Dimensionalität. Daher wird im folgenden Block der angepasste Standardscaler, der PCA und der gefundene Datensatz mit zu filternden Bildern gespeicher. Dies ermöglicht auch ab diesem Punkt weitere Tests und Änderungen vorzunehmen, ohne die vorangegangenen ressourcenaufwändigen Abläufe erneut durchlaufen zu müssen.

Die genannten Daten werden im Ordner "finishedData" gespeichert.

In [ ]:
outputPath = Path().cwd() / 'finishedData'
with open(outputPath / 'scaler.pkl','wb') as f:
    pickle.dump(scaler,f)
with open(outputPath / 'pca.pkl','wb') as f:
    pickle.dump(pca,f)
with open(outputPath / 'filteredPictures.txt','w') as f:
    for name in namesFilteredV2:
        f.write(f'{name}\n')        

Die Trainingsdaten werden ebenso in komprimierter Form gespeichert. Der dafür genutzte Ordner heißt "TrainTestSplit".

In [ ]:
saveDataPath = Path().cwd() / 'TrainTestSplit' / 'train_test_split.npz'

np.savez_compressed(
    saveDataPath,
    XTrain=XTrain,
    XTest = XTest,
    yTrain = yTrain,
    yTest = yTest
)

print(f'Data saved to {saveDataPath.resolve()}')
del(saveDataPath)

Data saved to C:\Users\max\OneDrive\Desktop\KIT\Semester 3\Seminar_KiP\Pitting_Detection\TrainTestSplit\train_test_split.npz


Hier werden erneut die ab diesem Punkt nicht mehr benötigten Daten aus dem Speicher gelöscht, um Ressourcen zu sparen.

In [50]:
del(ims, images, labels, X, y, X_scaled, X_pca, namesFilteredV2, pathsFiltered)

Dieser Block lädt den gespeicherten Datensatz, um einen einfachen Einstieg ab diesem Punkt zu ermöglichen. Wichtig zu erwähnen ist, dass hier immer der gleiche Datensatz geladen wird. Es werden also nicht zufällige Zuweisungen zu den Trainings- oder Testdaten durchgeführt.

In [3]:
loadDataPath = Path().cwd() / 'finishedData' / 'train_test_split.npz'

data = np.load(loadDataPath)

XTrain = data['XTrain']
XTest = data['XTest']
yTrain = data['yTrain']
yTest = data['yTest']

print('Data loaded successfully')
del(data)

Data loaded successfully


Im folgenden Block wird eine Funktion implementiert, die genutzt wird, um die Gridsearch durchführen zu können. Dafür werden die Daten und das genutzte Modell übergeben werden. Je nach Systemmöglichkeiten kann die nJobs variable angepasst werden, um parallele Berechnungen durchzuführen.
Nachdem das beste Modell gefunden ist, wird der dazu gehörige Report und die genutzten Parameter ausgegeben, sowie das erstellte Modell zurückgegeben.
Als gesuchte Metrik ist accuracy gewählt, da sonst ein einseitiges Training möglich ist. 

In [ ]:
def trainAndEvaluateModel(X,y, classifier, paramGrid, cvFold=5, scoring='accuracy', nJobs = 1, verbose=2):
    XTrain, XTest = X
    yTrain, yTest = y

    # Define pipeline
    pipe = Pipeline([('clf', classifier)])

    # Cross-validation strategy
    cv = StratifiedKFold(n_splits=cvFold, shuffle=True, random_state=42)

    # Grid search
    gridSearch = GridSearchCV(
        pipe,
        param_grid=paramGrid,
        cv=cv,
        scoring=scoring,
        n_jobs=nJobs,
        verbose=verbose
    )
    gridSearch.fit(XTrain, yTrain)

    # Best model
    best_model = gridSearch.best_estimator_
    print("Best Parameters:", gridSearch.best_params_)

    # Test set evaluation
    yPred = best_model.predict(XTest)
    report = classification_report(yTest, yPred)
    print("\nClassification Report (Test Set):\n", report)

    return best_model, report

Dieser Block speichert die Parameter für die jeweilige Untersuchung ab. Diese werden als Variable der Funktion übergeben und erleichtern den Aufruf.

In [36]:
rf_params = {
    'clf__n_estimators': [100, 300, 500],
    'clf__max_depth': [None, 25, 50],
    'clf__min_samples_split': [2, 5, 10],
    'clf__min_samples_leaf': [1, 3, 7]
}

gb_params = {
    'clf__n_estimators': [500],
    'clf__learning_rate': [0.1, 0.2],
    'clf__max_depth': [10, 30],
    'clf__min_samples_split': [2, 5],
    'clf__min_samples_leaf': [1, 3]
}

In [31]:
rfModel, rfReport = trainAndEvaluateModel((XTrain,XTest),(yTrain,yTest),RandomForestClassifier(random_state=42), rf_params)

Fitting 5 folds for each of 81 candidates, totalling 405 fits
[CV] END clf__max_depth=None, clf__min_samples_leaf=1, clf__min_samples_split=2, clf__n_estimators=100; total time=  11.8s
[CV] END clf__max_depth=None, clf__min_samples_leaf=1, clf__min_samples_split=2, clf__n_estimators=100; total time=  11.1s
[CV] END clf__max_depth=None, clf__min_samples_leaf=1, clf__min_samples_split=2, clf__n_estimators=100; total time=  11.3s
[CV] END clf__max_depth=None, clf__min_samples_leaf=1, clf__min_samples_split=2, clf__n_estimators=100; total time=  11.5s
[CV] END clf__max_depth=None, clf__min_samples_leaf=1, clf__min_samples_split=2, clf__n_estimators=100; total time=  11.5s
[CV] END clf__max_depth=None, clf__min_samples_leaf=1, clf__min_samples_split=2, clf__n_estimators=300; total time=  35.5s
[CV] END clf__max_depth=None, clf__min_samples_leaf=1, clf__min_samples_split=2, clf__n_estimators=300; total time=  34.2s
[CV] END clf__max_depth=None, clf__min_samples_leaf=1, clf__min_samples_split

In [37]:
gbModel, gbReport = trainAndEvaluateModel((XTrain,XTest),(yTrain,yTest),GradientBoostingClassifier(random_state=42), gb_params, nJobs=10)

Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best Parameters: {'clf__learning_rate': 0.2, 'clf__max_depth': 10, 'clf__min_samples_leaf': 3, 'clf__min_samples_split': 2, 'clf__n_estimators': 500}

Classification Report (Test Set):
               precision    recall  f1-score   support

       False       0.91      0.90      0.90       659
        True       0.91      0.92      0.91       729

    accuracy                           0.91      1388
   macro avg       0.91      0.91      0.91      1388
weighted avg       0.91      0.91      0.91      1388



Die folgenden beiden Codeblöcke bilden das Training des Klassifikators mit den optimalen Parametern und das darauf folgende Abspeichern des trainierten Modells ab. Nötig ist dieser Block, um das Training auf den gesamten Datensatz durchzuführen.

In [ ]:
clf = RandomForestClassifier(n_estimators=500, max_depth = 25, min_samples_leaf=1, min_samples_split=2, random_state=42)
clf.fit(XTrain, yTrain)
yPred = clf.predict(XTest)
print(classification_report(yTest,yPred,digits=4))

outputPath= Path().cwd() / 'trainedModels' / 'decisionTreeUnsimplified.pkl'
with open(outputPath,'wb') as f:
    pickle.dump(clf,f)

In [ ]:
clfXGB = XGBClassifier(
    n_estimators=500,
    max_depth=10,
    learning_rate=0.2,
    tree_method='hist',
    n_jobs=16,
    random_state=42,

    #ungetestete Empfehlungen
    subsample=0.8,            
    colsample_bytree=0.8,
)
clfXGB.fit(XTrain, yTrain)
yPredXGB = clfXGB.predict(XTest)
print(classification_report(yTest, yPredXGB, digits=4))

outputPath = Path().cwd()/'trainedModels'/'XGBoost.pkl'
with open('XGBBoost.pkl','wb') as f:
    pickle.dump(clfXGB,f)

              precision    recall  f1-score   support

       False       0.93      0.89      0.91      1423
        True       0.90      0.93      0.92      1545

    accuracy                           0.91      2968
   macro avg       0.92      0.91      0.91      2968
weighted avg       0.91      0.91      0.91      2968



Dieser Codeblock wurde genutzt, um die Reportzahlen auf 4 Nachkommastellen auszugeben um diese mit den CNN Ergebnisssen vergleichen zu können.

In [7]:
path = Path().cwd()/'trainedModels'/'XGBBoostUnsimplified.pkl'
with open (path,'rb') as f:
    clfXGB = pickle.load(f)
yPredXGB = clfXGB.predict(XTest)
print(classification_report(yTest, yPredXGB, digits=4))

              precision    recall  f1-score   support

       False     0.9263    0.8925    0.9091      1423
        True     0.9042    0.9346    0.9192      1545

    accuracy                         0.9144      2968
   macro avg     0.9153    0.9136    0.9141      2968
weighted avg     0.9148    0.9144    0.9143      2968

